## Task Objective

In this notebook I train a **YOLO object-detection model** that detects **fire** and **smoke** in images and videos.
This simulates a real-world safety-monitoring system: a camera watches a place, and the model raises an alert
when it sees fire or smoke, so people can react before the situation becomes dangerous.

## Dataset Description

| Item | Value |
|---|---|
| Source | Roboflow Universe |
| Workspace | `gfo-fc1de` |
| Project | `fire-and-smoke-detector-yykya` |
| Version | `14` |
| Format | YOLO (Ultralytics) |
| Link | https://universe.roboflow.com/gfo-fc1de/fire-and-smoke-detector-yykya/dataset/14 |

The dataset contains labeled images where every fire or smoke region is surrounded by a bounding box.

## Notebook Roadmap
1. Prepare the environment
2. Download the dataset
3. Train a YOLO detection model
4. Inspect the training results
5. Run prediction on a video
6. Final observations and limitations

## Step 1: Download the Dataset

I connect to Roboflow with my private API key (it is requested with a hidden input box, so the key is never saved
inside the notebook), choose the workspace, project and version, and download the dataset in **YOLO** format.

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="JMy8627CEJtSvlK5pED1")
project = rf.workspace("gfo-fc1de").project("fire-and-smoke-detector-yykya")
version = project.version(14)
dataset = version.download("yolo26")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 94.8 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to fire-and-smoke-detector-14 in yolo26:: 100%|██████████| 11204/11204 [00:01<00:00, 6066.51it/s]


## Step 2: Prepare the Environment

I install and import the required libraries:

- **Ultralytics**: provides the YOLO models plus training, validation and prediction utilities.
- **Roboflow**: downloads the dataset directly in YOLO format.
- **OpenCV**: reads and processes video files.
- **Matplotlib / Pandas / NumPy**: plotting images and reading the training metrics.


In [2]:
pip install Ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [Ultralytics]


In [3]:
pip install opencv-python

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from pathlib import Path

### Dataset structure and `data.yaml`
 the folders, the number of images in every split, and the `data.yaml` file.

### 📝 Explanation: YOLO dataset format

**What a YOLO dataset contains**
- An **`images/`** folder with the pictures.
- A **`labels/`** folder with one `.txt` file for every image (same file name).
  Each line describes one object:

  `class_id  x_center  y_center  width  height`

  All four box values are **normalized** between 0 and 1 (divided by the image width / height), so the labels still
  work if the image is resized.
- A **`data.yaml`** file that describes the dataset.

**The role of `data.yaml`**
It is the configuration file that tells YOLO where to find the `train`, `val` and `test` images, how many classes
there are (`nc`) and what their names are (`names`: fire, smoke). Training is started by pointing to this file.

**Why train, validation and test folders matter**
- **Train**: the images the model learns from (its weights are updated on them).
- **Validation**: images the model never trains on; used after every epoch to measure real performance, to choose
  `best.pt` and to detect overfitting.
- **Test**: kept completely untouched until the end to give an honest final score on unseen data.

Without this separation, the model could simply memorize the training images and we would wrongly believe it is good.

## Step 3: Train a YOLO Detection Model

Training settings (as required by the assignment):

| Parameter | Value |
|---|---|
| Task | Detection |
| Model | YOLO **small** (`yolo11s.pt`, pretrained on COCO) |
| Image size | 640 |
| Epochs | 60|
| Dataset config | `data.yaml` |

> 💡 If your Ultralytics version supports a newer small model (for example `yolo26s.pt`), you can change
> `MODEL_NAME` below. Everything else stays the same.

In [5]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26s.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="/content/fire-and-smoke-detector-14/data.yaml", epochs=60, imgsz=640)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fire-and-smoke-detector-14/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript,

### 📝 Explanation: training

**What detection means in YOLO**
Object detection answers two questions at once for every object in the image: **what is it** (class: fire or smoke)
and **where is it** (a bounding box). YOLO ("You Only Look Once") does this in a **single forward pass** through the
network, which makes it fast enough for real-time video.

**Why we start with a small YOLO model**
A nano / small model trains quickly, needs little GPU memory and runs in real time even on weak devices (a camera
or a Raspberry Pi). It is also a good baseline: if it already performs well we do not need a bigger model, and if not,
we know how much we have to improve.

**What epochs mean**
One epoch = the model has seen **every training image once**. With 60 epochs the whole training set is passed through
the network 60 times, and the weights improve a little in each pass. Too few epochs → underfitting; too many →
overfitting.

**What image size means**
`imgsz=640` means every image is resized (with padding to keep the aspect ratio) to 640×640 pixels before entering the
network. Bigger sizes keep more detail (useful for small objects such as distant smoke) but need more memory and time;
smaller sizes are faster but may miss small objects.

**What happens during model training**
1. A batch of images is loaded (with augmentation such as flips, color changes and mosaic).
2. The model predicts boxes, classes and confidence.
3. A **loss** is computed from the difference between predictions and ground truth (box loss, classification loss
   and distribution-focal loss).
4. **Backpropagation** calculates the gradients and the optimizer updates the weights.
5. After each epoch the model is evaluated on the validation set (precision, recall, mAP), and the best weights are saved.

In [6]:

# Load a model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # mAP50-95 per class; classes absent from the val set hold the overall mAP
metrics.box.ap_class_index  # class indices that were actually evaluated
metrics.box.image_metrics  # per-image metrics dictionary with precision, recall, F1, TP, FP, and FN

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,465,954 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1467.8±661.0 MB/s, size: 43.6 KB)
val: Scanning /content/fire-and-smoke-detector-14/valid/labels.cache... 1017 images, 105 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 1017/1017 164.1Mit/s 0.0s
val: /content/fire-and-smoke-detector-14/valid/images/WaterMistFireDemonstration3447_png.rf.8b898bfa25fa838fbdea67cf21cd9556.jpg: ignoring corrupt image/label: labels mix segment and detection rows
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 3.3it/s 19.3s
                   all       1016       1651      0.675      0.545      0.586       0.28
                  fire        835       1030      0.735      0.688      0.727      0.362
                 smoke        420        621      0.615      0.403      0.445      0.19

{'00168_jpg.rf.e63ab884ccc6262ae74f6c87875e2cda.jpg': {'precision': 0.013333333333333334,
  'recall': 0.5,
  'f1': 0.025974025974025976,
  'tp': 1,
  'fp': 74,
  'fn': 1},
 '00315_jpg.rf.646e191e0d652ecb54f9fd938b809714.jpg': {'precision': 0.03614457831325301,
  'recall': 1.0,
  'f1': 0.06976744186046512,
  'tp': 3,
  'fp': 80,
  'fn': 0},
 '00348_jpg.rf.b3eeae61640859c3ca333b32d0a5e832.jpg': {'precision': 0.030927835051546393,
  'recall': 1.0,
  'f1': 0.06000000000000001,
  'tp': 3,
  'fp': 94,
  'fn': 0},
 '00353_jpg.rf.aff76aea23683a3fe51f4919dd1818e1.jpg': {'precision': 0.019230769230769232,
  'recall': 1.0,
  'f1': 0.03773584905660378,
  'tp': 1,
  'fp': 51,
  'fn': 0},
 '00377_jpg.rf.104b99cca229e41286c9a1ce06800efa.jpg': {'precision': 0.017094017094017096,
  'recall': 1.0,
  'f1': 0.033613445378151266,
  'tp': 2,
  'fp': 115,
  'fn': 0},
 '0080-2_jpg.rf.c221e43ea6cf0fbcdb7717cd3a493073.jpg': {'precision': 0.05555555555555555,
  'recall': 1.0,
  'f1': 0.10526315789473684,
  'tp':

### 📝 Explanation: results and metrics

**Difference between `best.pt` and `last.pt`**
- **`last.pt`**: the weights saved at the **end of the final epoch** (also used to resume interrupted training).
- **`best.pt`**: the weights from the epoch that achieved the **highest fitness** on the validation set
  (a weighted mix of mAP@0.5 and mAP@0.5:0.95).

The last epoch is not always the best one, because the model may start overfitting. So `best.pt` is normally the file used
for prediction and deployment.

**Precision**
Out of all the objects the model *predicted* as fire/smoke, how many were really fire/smoke?

`Precision = TP / (TP + FP)`

High precision = **few false alarms**.

**Recall**
Out of all the *real* fire/smoke objects in the images, how many did the model find?

`Recall = TP / (TP + FN)`

High recall = **few missed fires**. In a safety system, recall is especially important because missing a real fire is
more dangerous than a false alarm.

**mAP (mean Average Precision)**
For each class, Average Precision (AP) is the area under the Precision-Recall curve. mAP is the mean AP over all classes
(fire and smoke).
- **mAP@0.5**: a detection is correct if its box overlaps the real box with **IoU ≥ 0.5**.
- **mAP@0.5:0.95**: the average over IoU thresholds from 0.5 to 0.95 (stricter, rewards accurate boxes).


### The prediction image

In [11]:


# Predict with the model
results = model("/content/Fire.jpg",save=True)  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box


image 1/1 /content/Fire.jpg: 448x640 1 fire, 18.2ms
Speed: 1.6ms preprocess, 18.2ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict


### The prediction video

In [13]:


# Predict with the model
results = model("/content/Fire_02.mp4",save=True)  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/480) /content/Fire_02.mp4: 384x640 (no detections), 12.6ms
video 1/1 (frame 2/480) /content/Fire_02.mp4: 384x640 (no detections), 11.8ms
video 1/1 (frame 3/480) /content/Fire_02.mp4: 384x640 1 fire, 11.7ms
video 1/1 (frame 4/480) /content/Fire_02.mp4: 384x640 1 fire, 11.7ms
video 1/1 (frame 5/480) /content/Fire_02.mp4: 384x640 2 fires, 11.8ms
video 1/1 (frame 6/480) /content/Fire_02.mp4: 384x640 2 fires, 17.3ms
video 1/1 (frame 7/480) /

## Export
- Export a YOLO26n model to a different format like ONNX, CoreML, etc

In [14]:


# Export the model
model.export(format="onnx")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26s summary (fused): 120 layers, 9,465,954 parameters, 0 gradients, 20.8 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (19.4 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 278ms
Prepared 4 packages in 673ms
Installed 4 packages in 3.46s
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.1
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 4.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming 

'/content/runs/detect/train/weights/best.onnx'